In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# config
DATA_DIR = "Data"
ONET_FILE = os.path.join(DATA_DIR, "onet_tasks.csv")
EUROSTAT_FILE = os.path.join(DATA_DIR, "Eurostat_employment_isco.xlsx")

COUNTRIES = ["Belgium", "Spain", "Poland"]
TASKS_NRCA = ["t_4A2a4", "t_4A2b2", "t_4A4a1"]

In [ ]:
def load_eurostat_data(filepath, countries):
    sheets = pd.read_excel(filepath, sheet_name=None)
    df_list = []
    
    for name, df in sheets.items():
        if name.startswith("ISCO"):
            df['ISCO'] = int(name.replace("ISCO", ""))
            df_list.append(df)
            
    all_data = pd.concat(df_list, ignore_index=True)

    # calc totals and shares
    for country in countries:
        total_col = f"total_{country}"
        share_col = f"share_{country}"
        all_data[total_col] = all_data.groupby('TIME')[country].transform('sum')
        all_data[share_col] = all_data[country] / all_data[total_col]
        
    return all_data

def load_onet_data(filepath):
    task_data = pd.read_csv(filepath)
    task_data["isco08_1dig"] = task_data["isco08"].astype(str).str[0].astype(int)
    return task_data.groupby("isco08_1dig").mean().drop(columns=["isco08"])

eurostat_data = load_eurostat_data(EUROSTAT_FILE, COUNTRIES)
onet_data = load_onet_data(ONET_FILE)

In [ ]:
def standardize(series, weights):
    mean_val = np.average(series, weights=weights)
    var_val = np.average((series - mean_val)**2, weights=weights)
    return (series - mean_val) / np.sqrt(var_val)

# merge
combined = pd.merge(eurostat_data, onet_data, left_on='ISCO', right_on='isco08_1dig', how='left')

# standardize tasks per country
for country in COUNTRIES:
    share_col = f"share_{country}"
    for task in TASKS_NRCA:
        std_col = f"std_{country}_{task}"
        combined[std_col] = standardize(combined[task], combined[share_col])

In [ ]:
def plot_country_trend(df, country, y_col):
    plt.figure(figsize=(8, 4))
    plt.plot(df["TIME"], df[y_col], label=country)
    plt.title(f"NRCA Task Intensity: {country}")
    plt.xticks(range(0, len(df), 3), df["TIME"][::3], rotation=45)
    plt.legend()
    plt.tight_layout()
    plt.show()

for country in COUNTRIES:
    share_col = f"share_{country}"
    nrca_col = f"{country}_NRCA"
    
    # sum standardized tasks
    combined[nrca_col] = sum(combined[f"std_{country}_{task}"] for task in TASKS_NRCA)

    # standardize combined NRCA
    std_nrca_col = f"std_{country}_NRCA"
    combined[std_nrca_col] = standardize(combined[nrca_col], combined[share_col])
    
    # calc country-level mean
    multip_col = f"multip_{country}_NRCA"
    combined[multip_col] = combined[std_nrca_col] * combined[share_col]

    # aggregate and plot
    agg_df = combined.groupby("TIME")[multip_col].sum().reset_index()
    plot_country_trend(agg_df, country, multip_col)